In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

import xgboost as xgb

In [2]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')

In [3]:
# Fill missing values with zeros
df = df.fillna(0)

target = "fuel_efficiency_mpg"

# Exact split required by the homework
df_full_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=1
)

df_train, df_val = train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=1
)

# Separate target from features
y_train = df_train[target].values
y_val = df_val[target].values

X_train_dict = (
    df_train.drop(columns=[target])
    .to_dict(orient="records")
)

X_val_dict = (
    df_val.drop(columns=[target])
    .to_dict(orient="records")
)

# Convert dictionaries into feature matrices
dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(X_train_dict)
X_val = dv.transform(X_val_dict)

feature_names = dv.get_feature_names_out()

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)


def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))



Training shape: (6000, 16)
Validation shape: (2000, 16)


Q1

In [7]:
dt = DecisionTreeRegressor(max_depth=1)

dt.fit(X_train, y_train)

root_index = dt.tree_.feature[0]

root_feature = feature_names[root_index]

# Convert one-hot feature back to original column name
original_feature = root_feature.split("=", 1)[0]

print("Actual split feature:", root_feature)
print("Original feature:", original_feature)

Actual split feature: model_year
Original feature: model_year


q2

In [6]:

rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)

rmse = calculate_rmse(y_val, y_pred)

print("Validation RMSE:", round(rmse, 4))



Validation RMSE: 1.8332


q3

In [8]:

estimators = [10, 50, 100, 150]

q3_results = {}

for n in estimators:

    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)

    rmse = calculate_rmse(y_val, y_pred)

    q3_results[n] = rmse

    print(f"n_estimators={n}: RMSE={rmse:.3f}")

best_estimators = min(q3_results, key=q3_results.get)

print("Best n_estimators:", best_estimators)



n_estimators=10: RMSE=1.833
n_estimators=50: RMSE=1.774
n_estimators=100: RMSE=1.770
n_estimators=150: RMSE=1.769
Best n_estimators: 150


q4

In [9]:

depths = [10, 15, 20, 25]
estimators = [10, 50, 100, 150]

q4_results = {}

for depth in depths:

    scores = []

    for n in estimators:

        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)

        rmse = calculate_rmse(y_val, y_pred)

        scores.append(rmse)

    mean_rmse = np.mean(scores)

    q4_results[depth] = mean_rmse

    print(
        f"max_depth={depth}: "
        f"mean RMSE={mean_rmse:.3f}"
    )

best_depth = min(q4_results, key=q4_results.get)

print("Best max_depth:", best_depth)


max_depth=10: mean RMSE=1.754
max_depth=15: mean RMSE=1.784
max_depth=20: mean RMSE=1.787
max_depth=25: mean RMSE=1.786
Best max_depth: 10


q5

In [10]:

rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf.feature_importances_
})

# Group one-hot encoded features by their original column
importance_df["original_feature"] = (
    importance_df["feature"].str.split("=", n=1).str[0]
)

importance_by_feature = (
    importance_df
    .groupby("original_feature")["importance"]
    .sum()
    .sort_values(ascending=False)
)

print("\nAll feature importances:")
print(importance_by_feature)

choices = [
    "vehicle_weight",
    "horsepower",
    "acceleration",
    "engine_displacement"
]

print("\nQuestion 5 options:")
print(importance_by_feature.loc[choices].sort_values(ascending=False))

print(
    "Most important among the four:",
    importance_by_feature.loc[choices].idxmax()
)




All feature importances:
original_feature
model_year             0.296107
vehicle_weight         0.203313
fuel_type              0.193555
horsepower             0.078271
engine_displacement    0.061281
acceleration           0.054998
origin                 0.052103
drivetrain             0.033651
num_doors              0.019401
num_cylinders          0.007318
Name: importance, dtype: float64

Question 5 options:
original_feature
vehicle_weight         0.203313
horsepower             0.078271
engine_displacement    0.061281
acceleration           0.054998
Name: importance, dtype: float64
Most important among the four: vehicle_weight


q6

In [11]:
# Create DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

# Create watchlist
watchlist = [
    (dtrain, "train"),
    (dval, "validation")
]

q6_results = {}

for eta in [0.3, 0.1]:

    xgb_params = {
        "eta": eta,
        "max_depth": 6,
        "min_child_weight": 1,

        "objective": "reg:squarederror",
        "nthread": 8,

        "seed": 1,
        "verbosity": 1
    }

    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=100,
        evals=watchlist,
        verbose_eval=False
    )

    y_pred = model.predict(dval)

    rmse = calculate_rmse(y_val, y_pred)

    q6_results[eta] = rmse

    print(f"eta={eta}: Validation RMSE={rmse:.4f}")

best_eta = min(q6_results, key=q6_results.get)

print("Best eta:", best_eta)


eta=0.3: Validation RMSE=1.8265
eta=0.1: Validation RMSE=1.7248
Best eta: 0.1


In [12]:

print("Q1 - Root feature:", original_feature)
print("Q2 - Random Forest RMSE:", round(q2_rmse := calculate_rmse(y_val, RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1).fit(X_train, y_train).predict(X_val)), 4))
print("Q3 - Best n_estimators:", best_estimators)
print("Q4 - Best max_depth:", best_depth)
print("Q5 - Most important feature:", importance_by_feature.loc[choices].idxmax())
print("Q6 - Best eta:", best_eta)

Q1 - Root feature: model_year
Q2 - Random Forest RMSE: 1.8332
Q3 - Best n_estimators: 150
Q4 - Best max_depth: 10
Q5 - Most important feature: vehicle_weight
Q6 - Best eta: 0.1
